# Exploración de las viviendas de Barcelona

Usad este notebook para conocer los datos **antes** de escribir `SPECS.md`. Lo que encontréis va a la sección **Datos de entrada y problemas** de `SPECS.md`, y las ideas para tratarlo, a **Estrategias y experimentos**.

Aquí se mira y se piensa: nada de lo que hagáis en este notebook entra en el modelo. Las transformaciones que decidáis se escriben en `SPECS.md`, y el agente las implementa en `src/`, dentro del `Pipeline` (o en `limpiar_objetivo()`, si son sobre el precio), para que se apliquen igual en la validación cruzada, en la validación y en el test. Este notebook es el único sitio donde escribís código vosotros.

El significado de cada columna está en `DATOS.md`. Recordad: los barrios son reales, pero las viviendas y los precios son simulados.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ desde notebooks/

from src.datos import cargar_entrenamiento, cargar_test

entrenamiento = cargar_entrenamiento()
test = cargar_test()
print(f"Entrenamiento: {entrenamiento.shape[0]:,d} filas · Test: {test.shape[0]:,d} filas")

Entrenamiento: 6,237 filas · Test: 4,002 filas


## 1. Perfil de cada columna

In [2]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos, ceros, valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "ceros": (df == 0).sum(),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

perfil(entrenamiento)

,tipo,nulos,% nulos,ceros,distintos,ejemplo
fecha_venta,str,0,0.0,0,1355,17/02/2024
municipi,str,0,0.0,0,12,Barcelona
districte,str,1503,24.1,0,10,Sant Andreu
barri,str,1503,24.1,0,72,el Bon Pastor
direccion,str,183,2.9,0,4992,"Passeig de l'Havana, 16"
codigo_postal,float64,218,3.5,0,75,8030.0
latitud,float64,108,1.7,0,3225,41.43817
longitud,float64,108,1.7,0,3253,2.19852
distancia_centro_km,float64,0,0.0,0,161,6.2
ref_catastral,str,0,0.0,0,6084,9130140CM9461Z1396MC


**¿Pasa lo mismo en el test?** El modelo tendrá que tasar esas 4.002 viviendas, con sus nulos y sus rarezas.

In [6]:
pd.DataFrame({
    "% nulos entrenamiento": (entrenamiento.isna().mean() * 100).round(1),
    "% nulos test": (test.isna().mean() * 100).round(1),
}).dropna(how="all").query("`% nulos entrenamiento` > 0 or `% nulos test` > 0")

,% nulos entrenamiento,% nulos test
any_construccio,29.8,30.1
ascensor,15.0,14.7
barri,24.1,23.1
certificado_energetico,8.1,8.4
codigo_postal,3.5,3.0
direccion,2.9,2.8
districte,24.1,23.1
estado,6.8,7.3
latitud,1.7,1.6
longitud,1.7,1.6


## 2. La variable objetivo

In [7]:
px.histogram(entrenamiento, x="precio", nbins=100, template="simple_white",
             title="Precio de la operación", labels={"precio": "Precio (€)"})

- ¿Qué forma tiene la distribución? ¿Cuáles son los precios más bajos y los más altos? ¿Alguno es imposible?
- Repetid el histograma con el logaritmo del precio. ¿Qué cambia? ¿Qué implicaría para un modelo lineal?
- `estado_operacion` dice si la operación se cerró. ¿Qué significa el precio de un anuncio **retirado**? ¿Es comparable con el de una venta?

In [8]:
entrenamiento.groupby("estado_operacion").agg(
    filas=("precio", "size"),
    precio_mediano=("precio", "median"),
    euros_m2_mediano=("precio", lambda p: (p / entrenamiento.loc[p.index, "superficie_construida"].replace(0, np.nan)).median()),
)

,filas,precio_mediano,euros_m2_mediano
estado_operacion,,,
retirada,620,332550.0,4211.881188
vendida,5617,282000.0,3496.666667


In [9]:
positivos = entrenamiento["precio"] > 0
print(f"Precios ≤ 0 (fuera del gráfico): {(~positivos).sum()}")

px.histogram(entrenamiento[positivos].assign(log10_precio=np.log10(entrenamiento.loc[positivos, "precio"])),
             x="log10_precio", nbins=100, template="simple_white",
             title="Precio de la operación (escala logarítmica)",
             labels={"log10_precio": "log₁₀ precio (5 = 100.000 €, 6 = 1.000.000 €)"})

Precios ≤ 0 (fuera del gráfico): 0


In [12]:
bajos = entrenamiento["precio"] < 30_000
altos = entrenamiento["precio"] > 3_000_000
print(f"Precio < 30.000 €: {bajos.sum()} · Precio > 3.000.000 €: {altos.sum()} · Total: {(bajos | altos).sum()}")
entrenamiento.loc[bajos | altos].sort_values("precio")[["municipi", "barri", "tipo", "superficie_construida", "estado_operacion", "precio"]]

Precio < 30.000 €: 22 · Precio > 3.000.000 €: 4 · Total: 26


,municipi,barri,tipo,superficie_construida,estado_operacion,precio
index,,,,,,
5287,Barcelona,Vilapicina i la Torre Llobeta,piso,38.0,vendida,123.1
9559,Barcelona,Vilapicina i la Torre Llobeta,Piso,52.0,vendida,128.9
1820,Barcelona,Torre Baró,piso,71.0,vendida,129.6
4442,Barcelona,la Sagrera,Estudio,39.0,vendida,142.3
8914,Barcelona,Ciutat Meridiana,Piso,93.0,vendida,147.8
2214,Barcelona,Porta,piso,59.0,vendida,160.2
1527,Barcelona,NaN,Piso,89.0,vendida,209.3
4043,Barcelona,la Prosperitat,Piso,127.0,vendida,220.8
2141,Barcelona,la Verneda i la Pau,PISO,85.0,vendida,221.6


## 3. Preguntas de calidad para empezar

- ¿Hay superficies imposibles (0, enormes)? ¿Es siempre la construida mayor que la útil?
- ¿Qué valores toma `planta`? ¿Cómo la convertiríais en un número? ¿Qué es "Entlo" o "Pral"?
- ¿Qué valores toman `ascensor`, `parking` y `tipo`? ¿Cuántas formas distintas hay de decir lo mismo? ¿Qué significa "Opcional" en `parking`?
- ¿Qué significa un 0 en `terraza_m2`? ¿Y un vacío?
- ¿Hay años de construcción imposibles?
- ¿En qué formatos está `fecha_venta`?
- ¿Cuándo falta el barrio? ¿Siempre fuera de Barcelona?
- ¿Hay `ref_catastral` repetidas? ¿Es la misma operación dos veces o la misma vivienda vendida dos veces?
- `dias_en_venta`: ¿lo sabe el banco en el momento de tasar?
- ¿Cuántas operaciones tiene cada barrio y cada municipio? ¿Hay zonas del test que no aparezcan en el entrenamiento?

In [27]:
superficies = entrenamiento[["superficie_construida", "superficie_util"]].melt(var_name="variable", value_name="m2").dropna()

print(entrenamiento[["superficie_construida", "superficie_util"]].describe().round(1))

px.histogram(superficies, x="m2", color="variable", facet_row="variable", nbins=100, template="simple_white",
             title="Superficie construida y superficie útil", labels={"m2": "Superficie (m²)"},
             height=600).update_yaxes(matches=None).update_layout(showlegend=False)

       superficie_construida  superficie_util
count                 6237.0           4191.0
mean                    94.0             76.9
std                     57.4             34.0
min                      0.0             20.0
25%                     65.0             54.0
50%                     85.0             70.0
75%                    110.0             92.0
max                   1590.0            348.0


In [25]:
entrenamiento["planta"].value_counts(dropna=False)

planta
NaN           302
3º            248
2º            243
1º            231
4º            227
             ... 
10             12
Planta 9        8
10ª planta      8
Planta 10       4
10a             3
Name: count, Length: 71, dtype: int64

**Ejemplo:** los diez valores más frecuentes de `planta`

In [23]:
entrenamiento["parking"].value_counts(dropna=False)

parking
No          3956
Sí           815
NaN          626
Opcional     462
Incluido     378
Name: count, dtype: int64

In [24]:
entrenamiento["ascensor"].value_counts(dropna=False)

ascensor
Sí       1321
NaN       935
No        590
True      473
Si        462
S         451
si        442
1         419
SI        406
no        194
False     193
N         183
0         168
Name: count, dtype: int64

In [28]:
entrenamiento["tipo"].value_counts(dropna=False)

tipo
Piso       3090
piso       1062
PISO       1001
Ático       271
Estudio     147
estudio     138
Casa        128
Atico       124
Duplex      104
Dúplex      103
Chalet       69
Name: count, dtype: int64

**Ejemplo:** zonas del test que no están en el entrenamiento

In [26]:
zona_ent = entrenamiento["barri"].fillna(entrenamiento["municipi"])
zona_test = test["barri"].fillna(test["municipi"])
nuevas = ~zona_test.isin(zona_ent)
print(f"{nuevas.sum()} viviendas del test están en {zona_test[nuevas].nunique()} zonas sin operaciones en el entrenamiento")

4 viviendas del test están en 1 zonas sin operaciones en el entrenamiento


## 4. Relación con el precio

Antes de probar una transformación, mirad cómo se relaciona cada variable con el precio. Una relación curva, valores imposibles o unos pocos puntos muy alejados son pistas de qué estrategia probar.

In [29]:
px.scatter(entrenamiento, x="superficie_construida", y="precio", color="municipi", opacity=0.4,
           template="simple_white", title="Precio frente a superficie construida",
           labels={"superficie_construida": "Superficie construida (m²)", "precio": "Precio (€)"})

- ¿Qué le pasa a este gráfico con ejes logarítmicos (`log_x=True, log_y=True`)?
- Repetidlo coloreando por `tipo` o por `estado`.
- Cread una columna con los €/m² y comparadla entre barrios: ¿cuánto varía?

In [30]:
entrenamiento.select_dtypes("number").corr(method="spearman")["precio"].sort_values()

distancia_centro_km     -0.400045
codigo_postal           -0.341811
longitud                -0.225743
latitud                 -0.219132
dias_en_venta           -0.040259
any_construccio         -0.029850
poblacion_zona           0.018046
terraza_m2               0.165615
banos                    0.507856
habitaciones             0.593337
superficie_construida    0.702580
superficie_util          0.713250
precio                   1.000000
Name: precio, dtype: float64

- ¿Por qué usamos la correlación de Spearman y no la de Pearson? (Sesión 06.)
- Una correlación baja, ¿quiere decir que la variable no sirve para tasar?

## 5. La ubicación

In [31]:
con_coordenadas = entrenamiento.dropna(subset=["latitud", "longitud"])
px.scatter_map(con_coordenadas, lat="latitud", lon="longitud", color=np.log10(con_coordenadas["precio"].clip(lower=1e4)),
               opacity=0.6, zoom=10, height=600, map_style="carto-positron",
               title="Precio según la ubicación", labels={"color": "log₁₀ precio"})

- ¿Dónde están las viviendas caras? ¿Ves algún punto fuera de sitio?
- La ubicación está en varias columnas: `municipi`, `districte`, `barri`, las coordenadas y `distancia_centro_km`. ¿Cuál usaríais en un modelo lineal y por qué?

## 6. Ideas para características nuevas

Pensad como un tasador: ¿qué miraría que no esté directamente en una columna? Algunas pistas: la altura real de la vivienda, si hay que subir escaleras, la época del edificio, el momento de la venta.

Escribid cada idea en `SPECS.md` §4 con su hipótesis **antes** de probarla.

In [ ]:
# Vuestras ideas aquí